# Phân tích Chuyên sâu Hành vi Gian lận

Phân tích xu hướng theo thời gian, rủi ro theo thiết bị và nơi ở, đánh giá độ chính xác của các bộ quy tắc heuristic.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.append("..")
from src.visualization.visualize import plot_fraud_by_category, plot_monthly_trend

df = pd.read_parquet("../data/processed/clean_base.parquet")


### 1. Xu hướng gian lận theo tháng

In [ ]:
df.groupby("month").agg(
    total_apps=("fraud_bool", "count"),
    fraud_cases=("fraud_bool", "sum"),
    fraud_rate=("fraud_bool", lambda x: x.mean() * 100)
).reset_index()

### 2. Phân tích rủi ro theo Hệ điều hành và Tình trạng nhà ở

In [ ]:
plot_fraud_by_category(df, "device_os", "Tỷ lệ gian lận theo Hệ điều hành")
plot_fraud_by_category(df, "housing_status", "Tỷ lệ gian lận theo Tình trạng nhà ở")

### 3. Đánh giá bộ quy tắc Heuristic
So sánh độ chính xác và số lượng kích hoạt của các quy tắc đơn giản với tỷ lệ cơ sở 1.10%.

In [ ]:
rule1 = (df["device_os"] == "windows") & (df["housing_status"] == "BA") & (df["has_prev_address"] == 0)
rule2 = (df["name_email_similarity"] < 0.3) & (df["email_is_free"] == 1) & (df["foreign_request"] == 1)
rule3 = (df["has_prior_bank_account"] == 0) & (df["no_valid_phone"] == 1)

eval_list = []
for name, mask in [("Rule 1: Windows + Nhà ở BA + Chưa có địa chỉ cũ", rule1),
                   ("Rule 2: Khác tên email + Email miễn phí + Nước ngoài", rule2),
                   ("Rule 3: Chưa có tài khoản cũ + Không có SĐT", rule3)]:
    triggered = mask.sum()
    caught = df.loc[mask, "fraud_bool"].sum()
    precision = (caught / triggered * 100) if triggered > 0 else 0
    eval_list.append({
        "Quy tắc": name,
        "Số hồ sơ kích hoạt": int(triggered),
        "Gian lận bắt được": int(caught),
        "Độ chính xác (%)": round(precision, 2)
    })

pd.DataFrame(eval_list)